# Laplace–Beltrami PDE Solver on Curved Manifolds

This notebook solves the **heat**, **Schrödinger**, and **wave** equations on a
2D Riemannian manifold using the spectral `PDESolver`.  The Laplace–Beltrami
operator $\Delta_g$ is extracted from a `Metric` object and fed into the
solver as a `psiOp` pseudo-differential operator.

**Three coupled steps:**
1. Pick a metric from the catalogue (one cell to edit).
2. Choose a PDE type and run the solver.
3. Animate the solution either on the flat parameter domain or *on the 3-D
   embedding* of the manifold.

---

## Mathematical background

Given a Riemannian metric $g_{ij}$ in local coordinates $(x,y)$, the
Laplace–Beltrami operator acting on a scalar $u$ is

$$
\Delta_g u = \frac{1}{\sqrt{|g|}} \partial_i\!\left(\sqrt{|g|}\, g^{ij} \partial_j u\right).
$$

Its full symbol (principal + subprincipal) is

$$
\sigma(\Delta_g)(x,\xi) = \underbrace{g^{ij}\xi_i\xi_j}_{\text{principal}} +
  i\underbrace{\frac{1}{\sqrt{|g|}} \partial_i(\sqrt{|g|}\, g^{ij})\xi_j}_{\text{subprincipal}},
$$

and $\Delta_g$ acts in Fourier space as multiplication by $-\sigma(\Delta_g)$.

The three PDEs are:

| Equation | Formulation | Operator symbol |
|---|---|---|
| Heat | $\partial_t u = \Delta_g u$ | $-\sigma(\Delta_g)$ |
| Schrödinger | $i\partial_t u = -\Delta_g u$ | $-i\,\sigma(\Delta_g)$ |
| Wave | $\partial_{tt} u = \Delta_g u$ | $-\sigma(\Delta_g)$ (2nd order) |

## 0. Imports

In [ ]:
import numpy as np
import sympy as sp
from sympy import symbols, Matrix, I
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML

from riemannian import Metric, build_embedding, plot_embedding
from solver import PDESolver
from psiop import *

import warnings
warnings.filterwarnings('ignore')

## 1. Metric catalogue

Pick one entry by setting `METRIC_NAME`.  Each entry is a dict with:

| key | meaning |
|---|---|
| `g` | 2×2 SymPy `Matrix` of metric components |
| `coords` | `(x, y)` coordinate symbols matching `g` |
| `domain` | `(Lx, Ly)` — physical **size** of the solver domain (centred at 0) |
| `shift` | `(x_shift, y_shift)` — offset added to solver coords before evaluating `g` |
| `ic_center` | centre of the Gaussian initial blob in **solver** coords |
| `ic_sigma` | width of the Gaussian |
| `Lt`, `Nt` | simulation time and number of time steps |
| `Nx`, `Ny` | spatial resolution |
| `description` | human-readable label |

> **Coordinate shift.**  The `PDESolver` always places its grid symmetrically
> around 0: $x\in[-L_x/2,L_x/2]$.  When the metric is ill-defined near 0
> (e.g. the Poincaré half-plane needs $y>0$), set `shift=(x0, y0)` so that
> the symbol is evaluated at the *shifted* coordinates $(x+x_0, y+y_0)$.

In [ ]:
# ── Coordinate symbols ───────────────────────────────────────────────────────
x, y = symbols('x y', real=True)


# ── Catalogue ────────────────────────────────────────────────────────────────
from metric_catalogue import *
# ── Select one ────────────────────────────────────────────────────────────────
METRIC_NAME = 'sphere' # ← change this

cfg = METRICS[METRIC_NAME]
print(f"Selected metric: {cfg['description']}")

boundary_condition = 'periodic'  # or 'dirichlet' or 'neumann' or 'periodic'

## 2. Build the `Metric` and inspect the Laplace–Beltrami symbol

We apply the coordinate shift here so that the symbol is expressed in terms
of the *solver* coordinates `(x_s, y_s) = (x - x_shift, y - y_shift)`.  The
solver evaluates the symbol on its own grid, which is centred at zero.

In [ ]:
# ── Build Metric ──────────────────────────────────────────────────────────────
metric = Metric(cfg['g'], cfg['coords'])

print(f"Dimension    : {metric.dim}")
print(f"Coordinates  : {metric.coords}")
print(f"Curvature K  : {sp.simplify(metric.gauss_curvature())}")

# ── Laplace–Beltrami symbol ───────────────────────────────────────────────────
lb = metric.laplace_beltrami_symbol()
print("\nPrincipal symbol  σ₂ =", lb['principal'])
print("Subprincipal symbol σ₁ =", lb['subprincipal'])
print("Full symbol  σ = σ₂ + i σ₁ =", lb['full'])

In [ ]:
# ── Apply coordinate shift to the symbol ─────────────────────────────────────
x_coord, y_coord = cfg['coords']
x_shift, y_shift = cfg['shift']

def shift_symbol(sym_expr):
    """Substitute (x → x + x_shift, y → y + y_shift) in a symbol expression."""
    subs = {}
    if x_shift != 0.0:
        subs[x_coord] = x_coord + x_shift
    if y_shift != 0.0:
        subs[y_coord] = y_coord + y_shift
    return sym_expr.subs(subs) if subs else sym_expr

sym_shifted = shift_symbol(lb['principal'])
full_sym_shifted = shift_symbol(lb['full'])
print("Shifted symbol:")
sp.pprint(sym_shifted)
print("Full shifted symbol:")
sp.pprint(full_sym_shifted)

In [ ]:
op = PseudoDifferentialOperator(sym_shifted, [x, y], mode='symbol')
domain = cfg['domain']
x_grid  = np.linspace(-domain[0]/2, domain[0]/2, 50)
y_grid  = np.linspace(-domain[1]/2, domain[1]/2, 50)
result = {
    'order':            op.symbol_order(),
    'principal_symbol': op.principal_symbol(),
    'formal_adjoint':   op.formal_adjoint(),
    'symplectic_flow':  op.symplectic_flow(),
    'is_self_adjoint':  op.is_self_adjoint(),
    'is_elliptic':      op.is_elliptic_numerically(x_grid=(x_grid, y_grid),
                                                    xi_grid=(x_grid, x_grid),),
}
print(result)


In [ ]:
op.interactive_symbol_analysis(
                            xlim=(-2, 2),
                            ylim=(-2, 2),
                            xi_range=(-2, 2),
                            eta_range=(-2, 2),
                            density=50)

## 3. PDE equation builders

Each builder returns a SymPy `Eq` that the `PDESolver` can parse.  The
relationship between the LBO symbol $\sigma$ and the `psiOp` argument is:

$$
\Delta_g u \;\leftrightarrow\; \mathtt{psiOp}(-\sigma,\, u)
$$

because the operator is defined with the **negative** symbol (the
positive-definite elliptic operator $-\Delta_g$ has positive principal
symbol $+g^{ij}\xi_i\xi_j$).

In [ ]:
def make_heat_eq(u_func, symbol_expr):
    """
    Heat equation:  ∂_t u = Δ_g u

    Parameters
    ----------
    u_func : applied SymPy Function, e.g. u(t, x, y)
    symbol_expr : full Laplace–Beltrami symbol (shifted if necessary)
    """
    t_var = u_func.args[0]
    return sp.Eq(sp.diff(u_func, t_var), psiOp(-symbol_expr, u_func))


def make_schrodinger_eq(u_func, symbol_expr):
    """
    Schrödinger equation:  i ∂_t u = -Δ_g u   ⟺   ∂_t u = i Δ_g u

    Parameters
    ----------
    u_func : applied SymPy Function, e.g. u(t, x, y)
    symbol_expr : full Laplace–Beltrami symbol (shifted if necessary)
    """
    t_var = u_func.args[0]
    return sp.Eq(sp.diff(u_func, t_var), psiOp(-I * symbol_expr, u_func))


def make_wave_eq(u_func, symbol_expr):
    """
    Wave equation:  ∂_tt u = Δ_g u

    Parameters
    ----------
    u_func : applied SymPy Function, e.g. u(t, x, y)
    symbol_expr : full Laplace–Beltrami symbol (shifted if necessary)
    """
    t_var = u_func.args[0]
    return sp.Eq(sp.diff(u_func, t_var, t_var), psiOp(-symbol_expr, u_func))


print("PDE builders ready.")

## 4. Shared simulation helpers

In [ ]:
# ── Symbolic function ─────────────────────────────────────────────────────────
t_sym = sp.Symbol('t', real=True, positive=True)
x_sym, y_sym = sp.symbols('x y', real=True)
u_sym = sp.Function('u')(t_sym, x_sym, y_sym)

# ── Grid parameters (from catalogue) ─────────────────────────────────────────
Lx, Ly     = cfg['domain']
Nx, Ny     = cfg['Nx'], cfg['Ny']
Lt, Nt     = cfg['Lt'], cfg['Nt']
x0_ic, y0_ic = cfg['ic_center']
sigma_ic   = cfg['ic_sigma']

# ── Initial conditions ────────────────────────────────────────────────────────
def gaussian_ic(X, Y):
    """Normalised 2-D Gaussian blob centred at (x0_ic, y0_ic)."""
    return np.exp(-((X - x0_ic)**2 + (Y - y0_ic)**2) / (2 * sigma_ic**2))

def zero_velocity(X, Y):
    """Zero initial velocity for the wave equation."""
    return np.zeros_like(X)

print(f"Domain : x ∈ [{-Lx/2:.2f}, {Lx/2:.2f}],  y ∈ [{-Ly/2:.2f}, {Ly/2:.2f}]")
print(f"Grid   : {Nx} × {Ny}, Lt = {Lt:.4f},  dt = {Lt/Nt:.4f}")

## 5. Solve – Heat equation

In [ ]:
eq_heat   = make_heat_eq(u_sym, sym_shifted)
slv_heat  = PDESolver(eq_heat)
slv_heat.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt,
    initial_condition=gaussian_ic,
    boundary_condition=boundary_condition,
    plot=False,
)
print("Solving heat equation …")
frames_heat = slv_heat.solve()
print(f"Done.  Stored {len(frames_heat)} frames.")

In [ ]:
ani_heat = slv_heat.animate(component='abs', mode='surface', overlay='front')
HTML(ani_heat.to_jshtml())

## 6. Solve – Schrödinger equation

In [ ]:
eq_schro  = make_schrodinger_eq(u_sym, sym_shifted)
slv_schro = PDESolver(eq_schro, time_scheme='default')
slv_schro.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt,
    initial_condition=gaussian_ic,
    boundary_condition=boundary_condition,
    plot=False,
)
print("Solving Schrödinger equation …")
frames_schro = slv_schro.solve()
print(f"Done.  Stored {len(frames_schro)} frames.")

In [ ]:
ani_schro = slv_schro.animate(component='abs', mode='surface', overlay='front')
HTML(ani_schro.to_jshtml())

## 7. Solve – Wave equation

In [ ]:
eq_wave  = make_wave_eq(u_sym, sym_shifted)
slv_wave = PDESolver(eq_wave)
slv_wave.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt,
    initial_condition=gaussian_ic,
    initial_velocity=zero_velocity,
    boundary_condition=boundary_condition,
    plot=False,
)
print("Solving wave equation …")
frames_wave = slv_wave.solve()
print(f"Done.  Stored {len(frames_wave)} frames.")

In [ ]:
ani_wave = slv_wave.animate(component='real', mode='surface', overlay='front')
HTML(ani_wave.to_jshtml())

## 8. 3-D embedding of the manifold

`build_embedding` constructs an approximate isometric embedding
$R : \Omega \to \mathbb{R}^3$ of the parameter domain into 3-D space by
marching row-by-row while maintaining a Darboux frame.  The result is a
numpy array of shape `(nu, nv, 3)`.

> **Note.** The parameter ranges passed to `build_embedding` must match the
> *actual* physical coordinates (i.e. the *shifted* grid).

In [ ]:
# Actual physical coordinate ranges (solver grid + shift)
u_range = (-Lx/2 + x_shift, Lx/2 + x_shift)
v_range = (-Ly/2 + y_shift, Ly/2 + y_shift)

print(f"Embedding parameter ranges:")
print(f"  u (= x_actual) ∈ [{u_range[0]:.3f}, {u_range[1]:.3f}]")
print(f"  v (= y_actual) ∈ [{v_range[0]:.3f}, {v_range[1]:.3f}]")

# Use the same resolution as the solver for a 1-to-1 pixel correspondence
R_embed, u_vals, v_vals = build_embedding(metric, u_range, v_range, Nx, Ny)
print(f"\nEmbedding shape: {R_embed.shape}   (nu={Nx}, nv={Ny}, xyz=3)")

In [ ]:
# ── Quick static view of the embedding ───────────────────────────────────────
plot_embedding(R_embed, title=f'Embedding — {cfg["description"]}', colormap='plasma', dark=False);

## 9. Animate the PDE solution *on the embedding*

The key idea is straightforward: the solver stores one `(Nx, Ny)` solution
array per frame, and the embedding `R` has shape `(Nx, Ny, 3)`.  At each
animation frame we colour the surface `(X, Y, Z)` with the corresponding
solution value — the geometry is fixed, only the colormap changes.

This works because the solver and the embedding share the **same parameter
grid** (both use `Nx × Ny` points over the same physical domain).

In [ ]:
def animate_on_embedding(
    R, frames, title='',
    component='abs', cmap='plasma',
    dark=True, interval=80, n_anim_frames=60,
):
    """
    Animate a scalar PDE solution on a 3-D embedding.

    Parameters
    ----------
    R : ndarray, shape (nu, nv, 3)
        The static 3-D embedding.  Its (nu, nv) grid must match that of
        every frame in `frames`.
    frames : list of ndarray, each shape (nu, nv)
        Solution snapshots (may be complex).  Produced by ``solver.solve()``.
    title : str
        Figure title.
    component : {'real', 'imag', 'abs', 'angle'}
        Which component of the (possibly complex) solution to display.
    cmap : str
        Matplotlib colormap name.
    dark : bool
        Dark (True) or light (False) background.
    interval : int
        Milliseconds between frames.
    n_anim_frames : int
        Number of animation frames (uniformly sampled from `frames`).

    Returns
    -------
    ani : matplotlib.animation.FuncAnimation
    """
    def _get_component(u):
        ops = {'real': np.real, 'imag': np.imag,
               'abs': np.abs,  'angle': np.angle}
        if component not in ops:
            raise ValueError(f"component must be one of {list(ops)}.")
        return ops[component](u)

    bg = '#111111' if dark else 'white'
    tc = 'white'   if dark else 'black'

    X, Y, Z = R[:,:,0], R[:,:,1], R[:,:,2]

    # Sample frames uniformly
    idx = np.linspace(0, len(frames) - 1, n_anim_frames, dtype=int)
    sampled = [_get_component(frames[i]) for i in idx]

    # Global colour scale for perceptual consistency across frames
    vmin = min(f.min() for f in sampled)
    vmax = max(f.max() for f in sampled)
    if vmax - vmin < 1e-12:
        vmin, vmax = 0.0, 1.0

    cm_func = plt.colormaps[cmap]

    fig = plt.figure(figsize=(8, 6), facecolor=bg)
    ax  = fig.add_subplot(111, projection='3d', facecolor=bg)
    ax.set_axis_off()

    def _norm(arr):
        return (arr - vmin) / (vmax - vmin)

    # Draw first frame
    surf = [ax.plot_surface(
        X, Y, Z,
        facecolors=cm_func(_norm(sampled[0])),
        linewidth=0, antialiased=True, shade=True,
    )]
    time_text = ax.text2D(
        0.02, 0.95, 't = 0.00',
        transform=ax.transAxes, color=tc, fontsize=10,
    )
    ax.set_title(title, color=tc, fontsize=11, pad=8)

    # Stable camera angle
    ax.view_init(elev=25, azim=-60)

    # Add a fixed colorbar (mappable with global scale)
    mappable = plt.cm.ScalarMappable(cmap=cmap)
    mappable.set_clim(vmin, vmax)
    cb = plt.colorbar(mappable, ax=ax, shrink=0.55, aspect=18, pad=0.02)
    cb.ax.yaxis.set_tick_params(color=tc, labelcolor=tc)
    cb.set_label(component, color=tc)

    def _update(frame_number):
        surf[0].remove()
        surf[0] = ax.plot_surface(
            X, Y, Z,
            facecolors=cm_func(_norm(sampled[frame_number])),
            linewidth=0, antialiased=True, shade=True,
        )
        t_val = (idx[frame_number] / (len(frames) - 1)) * slv_wave.Lt
        time_text.set_text(f't = {t_val:.2f}')
        return (surf[0],)

    ani = FuncAnimation(
        fig, _update,
        frames=n_anim_frames,
        interval=interval, blit=False,
    )
    plt.tight_layout()
    return ani

print("animate_on_embedding() defined.")

### 9a. Heat equation on the embedding

In [ ]:
ani_heat_3d = animate_on_embedding(
    R_embed, frames_heat,
    title=f'Heat equation — {cfg["description"]}',
    component='abs', cmap='inferno', dark=False
)
HTML(ani_heat_3d.to_jshtml())

### 9b. Schrödinger equation on the embedding

In [ ]:
ani_schro_3d = animate_on_embedding(
    R_embed, frames_schro,
    title=f'Schrödinger equation (|ψ|) — {cfg["description"]}',
    component='abs', cmap='viridis', dark=False
)
HTML(ani_schro_3d.to_jshtml())

### 9c. Wave equation on the embedding

In [ ]:
ani_wave_3d = animate_on_embedding(
    R_embed, frames_wave,
    title=f'Wave equation — {cfg["description"]}',
    component='abs', cmap='coolwarm', dark=False
)
HTML(ani_wave_3d.to_jshtml())

## 10. Side-by-side snapshot: parameter domain vs. embedding

Static comparison at the last stored frame, useful for publication figures.

In [ ]:
def snapshot_comparison(
    R, frames, solver, frame_idx=-1,
    component='abs', cmap='plasma', dark=True,
    pde_label='PDE',
):
    """
    Two-panel figure: flat domain (imshow) on the left,
    3-D embedding (surface) on the right.

    Parameters
    ----------
    R : ndarray (nu, nv, 3) — embedding
    frames : list of (nu, nv) arrays — solver frames
    solver : PDESolver — used only for grid info
    frame_idx : int — which frame to display (-1 = last)
    """
    def _get(u):
        ops = {'real': np.real, 'imag': np.imag,
               'abs': np.abs,  'angle': np.angle}
        return ops[component](u)

    bg = '#111111' if dark else 'white'
    tc = 'white'   if dark else 'black'

    scalar = _get(frames[frame_idx])
    X3, Y3, Z3 = R[:,:,0], R[:,:,1], R[:,:,2]

    vmin, vmax = scalar.min(), scalar.max()
    if vmax - vmin < 1e-12:
        vmin, vmax = 0.0, 1.0

    fig = plt.figure(figsize=(13, 5), facecolor=bg)

    # ── Left: flat imshow ────────────────────────────────────────────────────
    ax_flat = fig.add_subplot(1, 2, 1, facecolor=bg)
    extent  = [solver.x_grid[0], solver.x_grid[-1],
               solver.y_grid[0], solver.y_grid[-1]]
    im = ax_flat.imshow(
        scalar.T, origin='lower', extent=extent,
        cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto',
    )
    ax_flat.set_title(f'{pde_label} — parameter domain', color=tc)
    ax_flat.set_xlabel('x (solver)', color=tc)
    ax_flat.set_ylabel('y (solver)', color=tc)
    ax_flat.tick_params(colors=tc)
    for sp_ in ax_flat.spines.values():
        sp_.set_color('#444' if dark else 'gray')
    plt.colorbar(im, ax=ax_flat, shrink=0.85)

    # ── Right: 3-D embedding ─────────────────────────────────────────────────
    ax_3d = fig.add_subplot(1, 2, 2, projection='3d', facecolor=bg)
    s_norm = (scalar - vmin) / (vmax - vmin)
    ax_3d.plot_surface(
        X3, Y3, Z3,
        facecolors=plt.colormaps[cmap](s_norm),
        linewidth=0, antialiased=True, shade=True,
    )
    ax_3d.set_title(f'{pde_label} — embedding', color=tc)
    ax_3d.set_axis_off()
    ax_3d.view_init(elev=25, azim=-55)

    plt.suptitle(cfg['description'], color=tc, fontsize=12)
    plt.tight_layout()
    plt.show()


# ── Heat ──────────────────────────────────────────────────────────────────────
snapshot_comparison(R_embed, frames_heat, slv_heat,
                    component='abs', cmap='inferno',
                    pde_label='Heat equation')

# ── Schrödinger ───────────────────────────────────────────────────────────────
snapshot_comparison(R_embed, frames_schro, slv_schro,
                    component='abs', cmap='viridis',
                    pde_label='Schrödinger equation')

# ── Wave ──────────────────────────────────────────────────────────────────────
snapshot_comparison(R_embed, frames_wave, slv_wave,
                    component='real', cmap='coolwarm',
                    pde_label='Wave equation')

## 11. Semiclassical approach

Van Vleck–Pauli–Morette propagator for a point source:

$$
\psi(x,t) = \sum_{\text{rays}} \frac{1}{\sqrt{|\det J|}} \; e^{iS/\hbar - i\mu\pi/2}
$$

- $J = \partial x / \partial p_0$ – Jacobi matrix  
- $S = \int p\cdot\dot x\, dt$ – classical action  
- $\mu$ – Maslov index (caustic crossings)  

Caustics ($|\det J| \to 0$) are regularised with Airy / Pearcey functions.

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from propagator import compute_wavefunction, plot_wavefunction, plot_ray_fan, animate_wavefunction, plot_interference_detail
from propagator import EquationType

# Define 2D harmonic oscillator Hamiltonian
x, y, xi, eta = sp.symbols('x y xi eta', real=True)

# Source at origin
source = (0.0, 0.0)

# Momentum fan - exclude zero to avoid degenerate rays
p_mag = np.linspace(0.5, 2.0, 30)
p_ang = np.linspace(0, 2*np.pi, 40, endpoint=False)
p_fan = np.array([[p * np.cos(theta), p * np.sin(theta)] 
                  for p in p_mag for theta in p_ang])

print(f"Total rays: {len(p_fan)}")

### 11.1 Using the hamiltonian (Laplace-Beltrami operator)


Hamiltonian mode – solve $\partial^2_t u = \psi\mathrm{Op}(H, u)$ with  
$$
H(x,\xi) = g^{ij}(x)\,\xi_i\xi_j + i\,\underbrace{\frac{1}{\sqrt{|g|}}\partial_i\bigl(\sqrt{|g|}\,g^{ij}\bigr)\xi_j}_{\text{subprincipal}}.
$$

The principal symbol $g^{ij}\xi_i\xi_j$ (real) governs ray dynamics; the imaginary subprincipal term is **non‑classical** and should be omitted for a proper Hamiltonian.

In [ ]:
result = compute_wavefunction(
    hamiltonian = sym_shifted,
    coords      = (x, y),
    momenta     = (xi, eta),
    source      = source,
    p_fan       = p_fan,
    t_max       = 10.0,          # Less than π ≈ 3.14
    hbar        = 0.1,
    n_steps     = 200,
    N_grid      = 100,
    integrator  = 'rk45',
    equation    = EquationType.PARABOLIC,
    parallel    = True,
)
fig1 = plot_wavefunction(result, log_scale=False)
plt.show()

anim = animate_wavefunction(result, n_frames=80, interval=40)
HTML(anim.to_jshtml())

### 11.2 Using the metric associated to the Laplace-Beltrami operator


Metric mode – build the **real** kinetic Hamiltonian
$$
H = \frac12 g^{ij}(x)\,p_i p_j.
$$

Initial velocities $v^\alpha$ are converted to canonical momenta  
$p_\alpha = g_{\alpha\beta}(x_0)\,v^\beta$.  
The Schrödinger equation $i\partial_t\psi = \psi\mathrm{Op}(H,\psi)$ yields the semiclassical wavefunction above, with $\det J$ computed via the geodesic deviation equation.

In [ ]:
x, y = sp.symbols('x y', real=True)

vx_vals = np.linspace(-2.0, 2.0, 20)
vy_vals = np.linspace(-2.0, 2.0, 20)
v_fan_para = np.array([[vx, vy] for vx in vx_vals for vy in vy_vals])
 
result = compute_wavefunction(
    metric     = metric,
    source     = (0.0, 0.0),
    v_fan      = v_fan_para,
    t_max      = 1.0,
    hbar       = 0.01,
    n_steps    = 300,
    N_grid     = 200,
    integrator = 'verlet',
    equation   = EquationType.WAVE,
    parallel   = True,
)
 
fig5 = plot_wavefunction(result, log_scale=True)
plt.show()
 
fig5b = plot_ray_fan(result)
plt.show()

fig5c = plot_interference_detail(result)
plt.show()

anim = animate_wavefunction(result, n_frames=80, interval=40)
HTML(anim.to_jshtml())